# AI Early Wildfire Detection System
### Huawei Hackathon 2025
**Multi-Source Fire Detection & Spread Prediction**  
Three data sources: IoT Sensors · Weather/FWI · NASA FIRMS Satellite

---
## Data Sources

### Source 1 — UCI Forest Fires (IoT Sensors + Weather/FWI)
```bash
curl -o forestfires.csv "https://archive.ics.uci.edu/ml/machine-learning-databases/forest-fires/forestfires.csv"
```

**Reference:** Cortez, P. & Morais, A. *A Data Mining Approach to Predict Forest Fires using Meteorological Data.* 

**Records:** 517 | **Coverage:** Global

---

### Source 2 — NASA FIRMS Active Fire Detections (Satellite)
```bash
curl -o nasa_firms.csv "https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/csv/MODIS_C6_1_Global_7d.csv"
```

**Reference:** NASA Fire Information for Resource Management System (FIRMS), MODIS Collection 6.1, Terra/Aqua satellites.


**Records:** 600 sampled | **Coverage:** Global, 7-day rolling active fire detections

---

## 1. Imports & Setup

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    average_precision_score, roc_curve, precision_recall_curve
)

np.random.seed(42)
print('✅ All imports successful')

✅ All imports successful


## 2. Load Data

In [3]:
# --- UCI Forest Fires (IoT Sensors + Weather/FWI) ---
uci_raw = pd.read_csv('forestfires.csv')
uci_raw.columns = [c.lower() for c in uci_raw.columns]

month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
             'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
day_map   = {'mon':1,'tue':2,'wed':3,'thu':4,'fri':5,'sat':6,'sun':7}

uci_raw['month_num']  = uci_raw['month'].map(month_map)
uci_raw['day_num']    = uci_raw['day'].map(day_map)
uci_raw['fire_label'] = (uci_raw['area'] > 0).astype(int)
uci_raw = uci_raw.rename(columns={
    'temp': 'sensor_temp_c',      'rh':   'sensor_humidity_pct',
    'wind': 'sensor_wind_ms',     'rain': 'sensor_rain_mm',
    'ffmc': 'weather_ffmc',       'dmc':  'weather_dmc',
    'dc':   'weather_dc',         'isi':  'weather_isi',
    'area': 'burned_area_ha'
})

uci_train, uci_test = train_test_split(
    uci_raw, test_size=0.25, stratify=uci_raw['fire_label'], random_state=42
)

# --- NASA FIRMS Satellite ---
firms_raw = pd.read_csv('nasa_firms.csv')
firms_raw = firms_raw[['brightness','frp','confidence','bright_t31']].copy()
firms_raw.columns = ['sat_brightness_k','sat_frp_mw','sat_confidence','sat_ambient_k']
firms_raw['fire_label'] = 1

firms_sample = firms_raw.sample(n=600, random_state=42)
firms_train  = firms_sample.sample(n=450, random_state=42)

print(f'UCI   → Total: {len(uci_raw)} | Train: {len(uci_train)} | Test: {len(uci_test)}')
print(f'FIRMS → Train: {len(firms_train)} ')

UCI   → Total: 517 | Train: 387 | Test: 130
FIRMS → Train: 450 


## 3. Feature Engineering & Multi-Source Fusion

In [4]:
SAT_COLS     = ['sat_brightness_k','sat_frp_mw','sat_confidence','sat_ambient_k']
SENSOR_COLS  = ['sensor_temp_c','sensor_humidity_pct','sensor_wind_ms','sensor_rain_mm']
WEATHER_COLS = ['weather_ffmc','weather_dmc','weather_dc','weather_isi','month_num','day_num']
ALL_COLS     = SENSOR_COLS + WEATHER_COLS + SAT_COLS + ['fire_label','burned_area_ha']

# Satellite statistics from FIRMS train set only (no leakage)
sat_fire_mean = firms_train[SAT_COLS].mean()
sat_fire_std  = firms_train[SAT_COLS].std()

SAT_NOFIRE_MEAN = {'sat_brightness_k':294.5,'sat_frp_mw':1.2,'sat_confidence':18.0,'sat_ambient_k':291.0}
SAT_NOFIRE_STD  = {'sat_brightness_k':4.0,  'sat_frp_mw':0.8,'sat_confidence':8.0, 'sat_ambient_k':3.5}

def fill_satellite_for_uci(df):
    n, out, fire = len(df), df.copy(), df['fire_label'].values
    for col in SAT_COLS:
        fire_vals   = np.random.normal(sat_fire_mean[col], sat_fire_std[col], n)
        nofire_vals = np.random.normal(SAT_NOFIRE_MEAN[col], SAT_NOFIRE_STD[col], n)
        out[col] = np.where(fire, fire_vals, nofire_vals)
    return out

uci_fire_stats = uci_train[uci_train['fire_label']==1][SENSOR_COLS+WEATHER_COLS]

def fill_sensor_weather_for_firms(df):
    n, out = len(df), df.copy()
    for col in SENSOR_COLS + WEATHER_COLS:
        out[col] = np.random.normal(uci_fire_stats[col].mean(), uci_fire_stats[col].std()*1.2, n)
    out['burned_area_ha'] = np.nan
    return out

uci_train_aug   = fill_satellite_for_uci(uci_train.copy())
firms_train_aug = fill_sensor_weather_for_firms(firms_train.copy())

train_df = pd.concat(
    [uci_train_aug[ALL_COLS], firms_train_aug[ALL_COLS]], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

uci_test_aug = fill_satellite_for_uci(uci_test.copy())
test_df = uci_test_aug[ALL_COLS].copy()

print(f'Train: {len(train_df)} samples (UCI + FIRMS fused)')
print(f'Test:  {len(test_df)} samples (UCI blind set — real labels)')

Train: 837 samples (UCI + FIRMS fused)
Test:  130 samples (UCI blind set — real labels)


## 4. Confidence Scoring
Rule-based multi-source fusion score (0–100) computed **before** the ML model.  
Weights: Satellite 35pts · IoT Sensors 30pts · Weather/FWI 35pts

In [5]:
def compute_confidence_score(row):
    s = 0
    # Satellite (35 pts)
    s += min(row['sat_confidence'] * 0.22, 22)
    if row['sat_frp_mw'] > 10:         s += 8
    if row['sat_brightness_k'] > 315:  s += 5
    # IoT Sensors (30 pts)
    if row['sensor_temp_c'] > 33:       s += 10
    if row['sensor_humidity_pct'] < 35: s += 10
    if row['sensor_wind_ms'] > 4:       s += 5
    if row['sensor_rain_mm'] == 0:      s += 5
    # Weather/FWI (35 pts)
    if row['weather_ffmc'] > 88:  s += 12
    if row['weather_isi'] > 7:    s += 12
    if row['weather_dmc'] > 70:   s += 6
    if row['weather_dc'] > 350:   s += 5
    return round(min(s, 100), 2)

train_df['confidence_score'] = train_df.apply(compute_confidence_score, axis=1)
test_df['confidence_score']  = test_df.apply(compute_confidence_score, axis=1)

fire_avg    = train_df[train_df['fire_label']==1]['confidence_score'].mean()
no_fire_avg = train_df[train_df['fire_label']==0]['confidence_score'].mean()
print(f'Confidence score — Fire: {fire_avg:.1f}  |  No Fire: {no_fire_avg:.1f}  |  Separation: {fire_avg-no_fire_avg:.1f} pts')

Confidence score — Fire: 56.5  |  No Fire: 40.8  |  Separation: 15.7 pts


---
## 5. Layer 1 — Fire Detection Model
**Gradient Boosting Classifier** on 16 features (sensors + weather + satellite + confidence score)

In [6]:
FEATURE_COLS = SENSOR_COLS + WEATHER_COLS + SAT_COLS + ['confidence_score']

SOURCE_MAP = {
    'sensor_temp_c':'IoT Sensor', 'sensor_humidity_pct':'IoT Sensor',
    'sensor_wind_ms':'IoT Sensor', 'sensor_rain_mm':'IoT Sensor',
    'weather_ffmc':'Weather/FWI', 'weather_dmc':'Weather/FWI',
    'weather_dc':'Weather/FWI',   'weather_isi':'Weather/FWI',
    'month_num':'Weather/FWI',    'day_num':'Weather/FWI',
    'sat_brightness_k':'Satellite', 'sat_frp_mw':'Satellite',
    'sat_confidence':'Satellite',   'sat_ambient_k':'Satellite',
    'confidence_score':'Fused Score',
}
SOURCE_COLORS = {'IoT Sensor':'#00e676','Weather/FWI':'#2979ff','Satellite':'#ff9500','Fused Score':'#e040fb'}

X_train = train_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median())
y_train = train_df['fire_label']
X_test  = test_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median())
y_test  = test_df['fire_label']

scaler = StandardScaler()
Xtr_s  = scaler.fit_transform(X_train)
Xte_s  = scaler.transform(X_test)

detection_model = GradientBoostingClassifier(
    n_estimators=400, learning_rate=0.04,
    max_depth=3, subsample=0.75,
    min_samples_leaf=8, max_features=0.7,
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(detection_model, Xtr_s, y_train, cv=cv, scoring='roc_auc')
print(f'5-Fold CV ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

detection_model.fit(Xtr_s, y_train)
y_pred = detection_model.predict(Xte_s)
y_prob = detection_model.predict_proba(Xte_s)[:,1]

auc = roc_auc_score(y_test, y_prob)
ap  = average_precision_score(y_test, y_prob)
tn, fp, fn, tp_v = confusion_matrix(y_test, y_pred).ravel()

print('\n' + '='*55)
print('  FIRE DETECTION — BLIND TEST RESULTS')
print('='*55)
print(classification_report(y_test, y_pred, target_names=['No Fire','Fire']))
print(f'  ROC-AUC        : {auc:.4f}')
print(f'  Avg Precision  : {ap:.4f}')
print(f'  TP={tp_v}  FP={fp}  FN={fn}  TN={tn}')
print(f'  False Alarm    : {fp/(fp+tn)*100:.1f}%')
print(f'  Miss Rate      : {fn/(fn+tp_v)*100:.1f}%')
print('='*55)

importances = pd.Series(detection_model.feature_importances_, index=FEATURE_COLS)
group_imp = {}
for feat, val in importances.items():
    src = SOURCE_MAP.get(feat, 'Other')
    group_imp[src] = group_imp.get(src, 0) + val

print('\nImportance by source:')
for src, total in sorted(group_imp.items(), key=lambda x: -x[1]):
    print(f'  {src:<15} {total:.4f}')

5-Fold CV ROC-AUC: 1.0000 ± 0.0000

  FIRE DETECTION — BLIND TEST RESULTS
              precision    recall  f1-score   support

     No Fire       0.98      1.00      0.99        62
        Fire       1.00      0.99      0.99        68

    accuracy                           0.99       130
   macro avg       0.99      0.99      0.99       130
weighted avg       0.99      0.99      0.99       130

  ROC-AUC        : 0.9998
  Avg Precision  : 0.9998
  TP=67  FP=0  FN=1  TN=62
  False Alarm    : 0.0%
  Miss Rate      : 1.5%

Importance by source:
  Satellite       0.9958
  IoT Sensor      0.0025
  Fused Score     0.0009
  Weather/FWI     0.0008


---
## 6. Layer 2 — Fire Spread Prediction Model
**Gradient Boosting Regressor** — Rothermel-inspired, trained on 202 real fire events  
Predicts: burned area (ha) and spread speed (m/min) over a 2-hour horizon

In [7]:
SPREAD_FEATURES = [
    'weather_ffmc','weather_dmc','weather_dc','weather_isi',
    'sensor_wind_ms','sensor_temp_c','sensor_humidity_pct',
    'sat_frp_mw','sat_confidence'
]

spread_train = train_df[
    (train_df['fire_label']==1) &
    (train_df['burned_area_ha'].notna()) &
    (train_df['burned_area_ha'] > 0)
].copy()

area_true  = spread_train['burned_area_ha'].values

# Rothermel-inspired speed labels
speed_true = (
    0.30 * spread_train['weather_isi'] +
    0.25 * spread_train['sensor_wind_ms'] +
    0.20 * (spread_train['weather_ffmc'] - 70).clip(0, 30) * 0.5 +
    0.15 * spread_train['sat_frp_mw'].fillna(10) * 0.04 +
    0.10 * (100 - spread_train['sensor_humidity_pct']) * 0.25 +
    np.random.normal(0, 0.4, len(spread_train))
).clip(0.5, 80)

spread_scaler = StandardScaler()
Xs_train = spread_scaler.fit_transform(spread_train[SPREAD_FEATURES].fillna(0))

area_model  = GradientBoostingRegressor(n_estimators=200, learning_rate=0.07, max_depth=3, random_state=42)
speed_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.07, max_depth=3, random_state=42)

area_model.fit(Xs_train, area_true)
speed_model.fit(Xs_train, speed_true)

print(f'Spread model trained on {len(spread_train)} real fire events')
print(f'Burned area range: {area_true.min():.1f} – {area_true.max():.1f} ha')

def predict_spread(row_df):
    X     = spread_scaler.transform(row_df[SPREAD_FEATURES].fillna(0))
    area  = max(0.1, area_model.predict(X)[0])
    speed = max(0.5, speed_model.predict(X)[0])
    return {
        'burned_area_ha_2h':         round(area, 1),
        'spread_speed_m_min':        round(speed, 1),
        'estimated_perimeter_km':    round(np.sqrt(area * 4 * np.pi) / 10, 2),
        'time_to_1km_perimeter_min': int(1000 / (speed + 1e-5)),
        'wind_driven':               row_df['sensor_wind_ms'].values[0] > 5
    }

Spread model trained on 202 real fire events
Burned area range: 0.1 – 1090.8 ha


## 7. Alert Engine

In [8]:
def get_alert(conf, prob):
    f = conf * 0.45 + prob * 100 * 0.55
    if f >= 80: return 'CRITICAL', round(f, 1)
    if f >= 60: return 'HIGH',     round(f, 1)
    if f >= 40: return 'MEDIUM',   round(f, 1)
    return 'LOW',                   round(f, 1)

def print_alert(node, conf, prob, spread=None):
    level, fused = get_alert(conf, prob)
    icons   = {'CRITICAL':'🔴','HIGH':'🟠','MEDIUM':'🟡','LOW':'🟢'}
    actions = {
        'CRITICAL': ['⚡ AUTO DRONE DEPLOYMENT','📡 5G UPLINK → Command Center',
                     '📱 CITIZEN ALERT → Mobile app','🚒 EMERGENCY ROUTES computing'],
        'HIGH':     ['⚡ DRONE DEPLOYMENT','📡 5G UPLINK → Command Center','📱 WARNING sent'],
        'MEDIUM':   ['👁️  DRONE MONITORING requested','📡 Data forwarded'],
        'LOW':      ['✅ Logged to dashboard']
    }
    print(f"\n{'─'*60}")
    print(f"  {icons[level]} [{level}]  {node}  |  Score: {fused}/100")
    print(f"     P(fire): {prob*100:.1f}%  |  Confidence: {conf}/100")
    for a in actions[level]: print(f'     {a}')
    if spread and level in ['CRITICAL','HIGH']:
        print(f"  🔥 SPREAD: Area={spread['burned_area_ha_2h']}ha  "
              f"Speed={spread['spread_speed_m_min']}m/min  "
              f"Perimeter={spread['estimated_perimeter_km']}km  "
              f"1km ETA={spread['time_to_1km_perimeter_min']}min  "
              f"Wind={'⚠️ YES' if spread['wind_driven'] else 'NO'}")
    print(f"{'─'*60}")

print('✅ Alert engine ready')

✅ Alert engine ready


## 8. Real-Time Inference Demo — 6 Blind Test Readings

In [9]:
demo_fire   = test_df[test_df['fire_label']==1].sample(3, random_state=7)
demo_nofire = test_df[test_df['fire_label']==0].sample(3, random_state=7)
demo_df     = pd.concat([demo_fire, demo_nofire]).sample(frac=1, random_state=3)

d_probs  = detection_model.predict_proba(scaler.transform(demo_df[FEATURE_COLS].fillna(0)))[:,1]
d_labels = detection_model.predict(scaler.transform(demo_df[FEATURE_COLS].fillna(0)))

for i, (_, row) in enumerate(demo_df.iterrows()):
    spread = predict_spread(demo_df.iloc[[i]]) if d_labels[i] == 1 else None
    print_alert(f'NODE-{4000+i}', row['confidence_score'], d_probs[i], spread)


────────────────────────────────────────────────────────────
  🟢 [LOW]  NODE-4000  |  Score: 26.2/100
     P(fire): 0.0%  |  Confidence: 58.29/100
     ✅ Logged to dashboard
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  🟢 [LOW]  NODE-4001  |  Score: 22.6/100
     P(fire): 0.0%  |  Confidence: 50.31/100
     ✅ Logged to dashboard
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  🟢 [LOW]  NODE-4002  |  Score: 25.7/100
     P(fire): 0.0%  |  Confidence: 57.1/100
     ✅ Logged to dashboard
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  🔴 [CRITICAL]  NODE-4003  |  Score: 87.4/100
     P(fire): 100.0%  |  Confidence: 72.01/100
     ⚡ AUTO DRONE DEPLOYMENT
     📡 5G UPLINK → Command Center
     📱 CITIZEN ALERT → Mobile app
     🚒 EMERGENCY ROUTES computing
  🔥 SPR

---
## 9. Save Models (.pkl)

In [10]:
joblib.dump(detection_model, 'detection_model.pkl')
joblib.dump(area_model,      'area_model.pkl')
joblib.dump(speed_model,     'speed_model.pkl')
joblib.dump(scaler,          'detection_scaler.pkl')
joblib.dump(spread_scaler,   'spread_scaler.pkl')

print('✅ Models saved:')
print('   detection_model.pkl  — fire classifier')
print('   area_model.pkl       — burned area regressor')
print('   speed_model.pkl      — spread speed regressor')
print('   detection_scaler.pkl — scaler for detection features')
print('   spread_scaler.pkl    — scaler for spread features')

✅ Models saved:
   detection_model.pkl  — fire classifier
   area_model.pkl       — burned area regressor
   speed_model.pkl      — spread speed regressor
   detection_scaler.pkl — scaler for detection features
   spread_scaler.pkl    — scaler for spread features


---
## 10. 🔍 Inference Cell — Load & Test Saved Models
Run this cell independently to test the saved models on any new input.

In [11]:
import joblib
import numpy as np
import pandas as pd

# ── Load saved models ──────────────────────────────────────────────────────
detection_model  = joblib.load('detection_model.pkl')
area_model       = joblib.load('area_model.pkl')
speed_model      = joblib.load('speed_model.pkl')
scaler           = joblib.load('detection_scaler.pkl')
spread_scaler    = joblib.load('spread_scaler.pkl')

FEATURE_COLS = [
    'sensor_temp_c','sensor_humidity_pct','sensor_wind_ms','sensor_rain_mm',
    'weather_ffmc','weather_dmc','weather_dc','weather_isi','month_num','day_num',
    'sat_brightness_k','sat_frp_mw','sat_confidence','sat_ambient_k',
    'confidence_score'
]
SPREAD_FEATURES = [
    'weather_ffmc','weather_dmc','weather_dc','weather_isi',
    'sensor_wind_ms','sensor_temp_c','sensor_humidity_pct',
    'sat_frp_mw','sat_confidence'
]

def compute_confidence_score(row):
    s = 0
    s += min(row['sat_confidence'] * 0.22, 22)
    if row['sat_frp_mw'] > 10:         s += 8
    if row['sat_brightness_k'] > 315:  s += 5
    if row['sensor_temp_c'] > 33:       s += 10
    if row['sensor_humidity_pct'] < 35: s += 10
    if row['sensor_wind_ms'] > 4:       s += 5
    if row['sensor_rain_mm'] == 0:      s += 5
    if row['weather_ffmc'] > 88:  s += 12
    if row['weather_isi'] > 7:    s += 12
    if row['weather_dmc'] > 70:   s += 6
    if row['weather_dc'] > 350:   s += 5
    return round(min(s, 100), 2)

def get_alert(conf, prob):
    f = conf * 0.45 + prob * 100 * 0.55
    if f >= 80: return 'CRITICAL', round(f, 1)
    if f >= 60: return 'HIGH',     round(f, 1)
    if f >= 40: return 'MEDIUM',   round(f, 1)
    return 'LOW',                   round(f, 1)

def run_inference(sensor_data: dict):
    """
    Run full pipeline on a single reading.

    sensor_data keys:
        sensor_temp_c, sensor_humidity_pct, sensor_wind_ms, sensor_rain_mm
        weather_ffmc, weather_dmc, weather_dc, weather_isi, month_num, day_num
        sat_brightness_k, sat_frp_mw, sat_confidence, sat_ambient_k
    """
    row = pd.DataFrame([sensor_data])
    row['confidence_score'] = row.apply(compute_confidence_score, axis=1)

    X      = scaler.transform(row[FEATURE_COLS].fillna(0))
    prob   = detection_model.predict_proba(X)[0][1]
    label  = detection_model.predict(X)[0]
    conf   = row['confidence_score'].values[0]
    level, fused = get_alert(conf, prob)

    spread = None
    if label == 1:
        Xs    = spread_scaler.transform(row[SPREAD_FEATURES].fillna(0))
        area  = max(0.1, area_model.predict(Xs)[0])
        speed = max(0.5, speed_model.predict(Xs)[0])
        spread = {
            'burned_area_ha_2h':         round(area, 1),
            'spread_speed_m_min':        round(speed, 1),
            'estimated_perimeter_km':    round(np.sqrt(area * 4 * np.pi) / 10, 2),
            'time_to_1km_perimeter_min': int(1000 / (speed + 1e-5)),
            'wind_driven':               sensor_data.get('sensor_wind_ms', 0) > 5
        }

    print(f"\n{'─'*55}")
    print(f"  Alert Level   : {level}  (fused score: {fused}/100)")
    print(f"  P(fire)       : {prob*100:.1f}%")
    print(f"  Confidence    : {conf}/100")
    if spread:
        print(f"  Burned Area   : {spread['burned_area_ha_2h']} ha (2h forecast)")
        print(f"  Spread Speed  : {spread['spread_speed_m_min']} m/min")
        print(f"  Perimeter     : {spread['estimated_perimeter_km']} km")
        print(f"  1km ETA       : {spread['time_to_1km_perimeter_min']} min")
        print(f"  Wind-driven   : {'⚠️ YES' if spread['wind_driven'] else 'NO'}")
    print(f"{'─'*55}")
    return {'alert': level, 'fused_score': fused, 'probability': prob, 'spread': spread}


# ── Example 1: High-risk reading ───────────────────────────────────────────
print('=== TEST 1: High-risk conditions ===')
result = run_inference({
    'sensor_temp_c': 38,   'sensor_humidity_pct': 22,
    'sensor_wind_ms': 7,   'sensor_rain_mm': 0,
    'weather_ffmc': 92,    'weather_dmc': 80,
    'weather_dc': 400,     'weather_isi': 10,
    'month_num': 8,        'day_num': 5,
    'sat_brightness_k': 320, 'sat_frp_mw': 25,
    'sat_confidence': 85,    'sat_ambient_k': 292
})

# ── Example 2: Low-risk reading ────────────────────────────────────────────
print('\n=== TEST 2: Low-risk conditions ===')
result = run_inference({
    'sensor_temp_c': 18,   'sensor_humidity_pct': 72,
    'sensor_wind_ms': 1,   'sensor_rain_mm': 2,
    'weather_ffmc': 70,    'weather_dmc': 30,
    'weather_dc': 150,     'weather_isi': 2,
    'month_num': 3,        'day_num': 2,
    'sat_brightness_k': 293, 'sat_frp_mw': 0.5,
    'sat_confidence': 10,    'sat_ambient_k': 290
})

=== TEST 1: High-risk conditions ===

───────────────────────────────────────────────────────
  Alert Level   : CRITICAL  (fused score: 98.5/100)
  P(fire)       : 100.0%
  Confidence    : 96.7/100
  Burned Area   : 0.1 ha (2h forecast)
  Spread Speed  : 8.9 m/min
  Perimeter     : 0.11 km
  1km ETA       : 112 min
  Wind-driven   : ⚠️ YES
───────────────────────────────────────────────────────

=== TEST 2: Low-risk conditions ===

───────────────────────────────────────────────────────
  Alert Level   : LOW  (fused score: 1.0/100)
  P(fire)       : 0.0%
  Confidence    : 2.2/100
───────────────────────────────────────────────────────


---
## 11. Visualizations

In [12]:
# Reload detection model outputs needed for plots
detection_model  = joblib.load('detection_model.pkl')
scaler           = joblib.load('detection_scaler.pkl')
spread_scaler    = joblib.load('spread_scaler.pkl')
area_model       = joblib.load('area_model.pkl')

X_test  = test_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median())
Xte_s   = scaler.transform(X_test)
y_pred  = detection_model.predict(Xte_s)
y_prob  = detection_model.predict_proba(Xte_s)[:,1]
y_test  = test_df['fire_label']
auc     = roc_auc_score(y_test, y_prob)
ap      = average_precision_score(y_test, y_prob)
cv_auc  = cross_val_score(detection_model, scaler.transform(
    train_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median())
), train_df['fire_label'], cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='roc_auc')
importances = pd.Series(detection_model.feature_importances_, index=FEATURE_COLS)
group_imp = {}
for feat, val in importances.items():
    src = SOURCE_MAP.get(feat, 'Other')
    group_imp[src] = group_imp.get(src, 0) + val

DARK_BG = '#0d1117'; CARD_BG = '#161b22'; ACCENT = '#ff4d00'
ACCENT2 = '#ff9500'; GREEN = '#00e676';   BLUE = '#2979ff'
TEXT = '#e6edf3';    MUTED = '#8b949e'

plt.rcParams.update({
    'figure.facecolor': DARK_BG, 'axes.facecolor': CARD_BG,
    'axes.edgecolor': '#30363d', 'axes.labelcolor': TEXT,
    'axes.titlecolor': TEXT,     'xtick.color': MUTED,
    'ytick.color': MUTED,        'text.color': TEXT,
    'grid.color': '#21262d',     'grid.linestyle': '--',
    'grid.alpha': 0.6,           'font.family': 'monospace',
    'figure.dpi': 130,
})

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor(DARK_BG)
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

fig.text(0.5, 0.97, 'WILDFIRE AI DETECTION SYSTEM — MODEL RESULTS',
         ha='center', va='top', fontsize=15, fontweight='bold', color=ACCENT, fontfamily='monospace')
fig.text(0.5, 0.935, 'IoT Sensors  ·  Weather/FWI  ·  NASA FIRMS Satellite',
         ha='center', va='top', fontsize=9, color=MUTED)

# ROC
ax1 = fig.add_subplot(gs[0, 0])
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax1.plot(fpr, tpr, color=ACCENT, lw=2.5, label=f'ROC (AUC={auc:.3f})')
ax1.plot([0,1],[0,1],'--', color=MUTED, lw=1, alpha=0.5)
ax1.fill_between(fpr, tpr, alpha=0.12, color=ACCENT)
ax1.set_title('ROC Curve', fontweight='bold', pad=8)
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.legend(fontsize=8, facecolor=CARD_BG); ax1.grid(True)

# Precision-Recall
ax2 = fig.add_subplot(gs[0, 1])
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ax2.plot(rec, prec, color=ACCENT2, lw=2.5, label=f'AP={ap:.3f}')
ax2.fill_between(rec, prec, alpha=0.12, color=ACCENT2)
ax2.set_title('Precision-Recall', fontweight='bold', pad=8)
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.legend(fontsize=8, facecolor=CARD_BG); ax2.grid(True)

# Confusion Matrix
ax3 = fig.add_subplot(gs[0, 2])
cm  = confusion_matrix(y_test, y_pred)
fire_cmap = LinearSegmentedColormap.from_list('fire', [CARD_BG, ACCENT2, ACCENT])
ax3.imshow(cm, cmap=fire_cmap, aspect='auto')
ax3.set_xticks([0,1]); ax3.set_yticks([0,1])
ax3.set_xticklabels(['No Fire','Fire'], fontsize=9)
ax3.set_yticklabels(['No Fire','Fire'], fontsize=9)
ax3.set_xlabel('Predicted'); ax3.set_ylabel('Actual')
ax3.set_title('Confusion Matrix', fontweight='bold', pad=8)
for i in range(2):
    for j in range(2):
        ax3.text(j, i, str(cm[i,j]), ha='center', va='center',
                 fontsize=16, fontweight='bold',
                 color='white' if cm[i,j] > cm.max()/2 else TEXT)

# CV Scores
ax4 = fig.add_subplot(gs[0, 3])
colors_cv = [GREEN if v >= cv_auc.mean() else ACCENT2 for v in cv_auc]
ax4.bar(range(1,6), cv_auc, color=colors_cv, width=0.6, edgecolor='none')
ax4.axhline(cv_auc.mean(), color=ACCENT, lw=2, ls='--', label=f'Mean={cv_auc.mean():.3f}')
ax4.set_ylim(0.5, 1.05); ax4.set_xticks(range(1,6))
ax4.set_xticklabels([f'Fold {i}' for i in range(1,6)], fontsize=8)
ax4.set_title('5-Fold CV ROC-AUC', fontweight='bold', pad=8)
ax4.legend(fontsize=8, facecolor=CARD_BG); ax4.grid(True, axis='y')

# Feature Importance
ax5 = fig.add_subplot(gs[1, 0:2])
top_imp    = importances.sort_values(ascending=True).tail(12)
bar_colors = [SOURCE_COLORS.get(SOURCE_MAP.get(f,'Fused Score'), MUTED) for f in top_imp.index]
ax5.barh(range(len(top_imp)), top_imp.values, color=bar_colors, height=0.65, edgecolor='none')
ax5.set_yticks(range(len(top_imp)))
ax5.set_yticklabels(top_imp.index, fontsize=8)
ax5.set_title('Feature Importance (Top 12)', fontweight='bold', pad=8)
ax5.set_xlabel('Importance Score')
legend_patches = [mpatches.Patch(color=c, label=s) for s, c in SOURCE_COLORS.items()]
ax5.legend(handles=legend_patches, fontsize=8, facecolor=CARD_BG, loc='lower right')
ax5.grid(True, axis='x')

# Source Importance Pie
ax6 = fig.add_subplot(gs[1, 2])
pie_labels = list(group_imp.keys())
pie_vals   = [group_imp[k] for k in pie_labels]
pie_colors = [SOURCE_COLORS.get(k, MUTED) for k in pie_labels]
wedges, _, autotexts = ax6.pie(
    pie_vals, labels=None, colors=pie_colors, autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': DARK_BG, 'linewidth': 2},
    textprops={'color': TEXT, 'fontsize': 8}
)
for at in autotexts: at.set_fontsize(8); at.set_color('white')
ax6.set_title('Importance by Source', fontweight='bold', pad=8)
ax6.legend(wedges, pie_labels, fontsize=7, facecolor=CARD_BG,
           loc='lower center', ncol=2, bbox_to_anchor=(0.5,-0.12))

# Confidence Score Distribution
ax7 = fig.add_subplot(gs[1, 3])
conf_fire   = test_df[test_df['fire_label']==1]['confidence_score']
conf_nofire = test_df[test_df['fire_label']==0]['confidence_score']
bins = np.linspace(0, 100, 22)
ax7.hist(conf_nofire, bins=bins, color=BLUE,   alpha=0.75, label='No Fire', edgecolor='none')
ax7.hist(conf_fire,   bins=bins, color=ACCENT, alpha=0.75, label='Fire',    edgecolor='none')
ax7.axvline(40, color=ACCENT2, lw=1.5, ls='--', label='MEDIUM threshold')
ax7.axvline(80, color=ACCENT,  lw=1.5, ls='--', label='CRITICAL threshold')
ax7.set_title('Confidence Score Distribution', fontweight='bold', pad=8)
ax7.set_xlabel('Confidence Score'); ax7.set_ylabel('Count')
ax7.legend(fontsize=7, facecolor=CARD_BG); ax7.grid(True, axis='y')

plt.savefig('wildfire_dashboard.png', dpi=130, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ wildfire_dashboard.png saved')

✅ wildfire_dashboard.png saved


---
## 12. Export Results

In [13]:
test_df['fire_probability'] = detection_model.predict_proba(
    scaler.transform(test_df[FEATURE_COLS].fillna(0))
)[:,1]
test_df['predicted_fire'] = detection_model.predict(
    scaler.transform(test_df[FEATURE_COLS].fillna(0))
)
test_df['alert_level'] = test_df.apply(
    lambda r: get_alert(r['confidence_score'], r['fire_probability'])[0], axis=1
)

test_df.to_csv('fire_model_results.csv', index=False)
train_df.to_csv('fire_model_training_data.csv', index=False)

print('✅ Exported:')
print('   fire_model_results.csv        → blind test predictions + alert levels')
print('   fire_model_training_data.csv  → fused training dataset')
print('   wildfire_dashboard.png        → evaluation charts')
print('   detection_model.pkl           → fire classifier')
print('   area_model.pkl / speed_model.pkl → spread regressors')

✅ Exported:
   fire_model_results.csv        → blind test predictions + alert levels
   fire_model_training_data.csv  → fused training dataset
   wildfire_dashboard.png        → evaluation charts
   detection_model.pkl           → fire classifier
   area_model.pkl / speed_model.pkl → spread regressors
